# FactChecker AI - DeBERTa Fine-Tuning
Runtime: T4 GPU required
Before running: add HF_TOKEN to Colab secrets (key icon, left sidebar)

In [ ]:
%pip install -q transformers datasets accelerate evaluate scikit-learn pandas numpy huggingface_hub sentencepiece protobuf
print('Done')

In [ ]:
import torch
from huggingface_hub import login
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    login(token=HF_TOKEN)
    print('HuggingFace login OK')
except Exception as e:
    print(f'HF login failed: {e}')
    HF_TOKEN = None

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

HF_USERNAME = 'Bharat2004'
MODEL_NAME  = f'{HF_USERNAME}/factchecker-deberta'
print(f'Will push to: {MODEL_NAME}')

In [ ]:
import pandas as pd
import numpy as np
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

frames = []

# Dataset 1: GonzaloA/fake_news (24k, confirmed working)
print('Loading GonzaloA/fake_news...')
try:
    ds = load_dataset('GonzaloA/fake_news', split='train')
    df = ds.to_pandas()
    df['text'] = (df.get('title', pd.Series(['']*len(df))).fillna('') + ' ' + df['text'].fillna('')).str.strip()
    df['label'] = df['label'].astype(int)
    frames.append(df[['text','label']])
    print(f'  OK: {len(df)} rows, labels: {df["label"].value_counts().to_dict()}')
except Exception as e:
    print(f'  Failed: {e}')

# Dataset 2: Arko007/fact-check1-v1 training data (if accessible)
print('Loading Yakoob-Khan/Fake-News-Detection...')
try:
    ds = load_dataset('Yakoob-Khan/Fake-News-Detection', split='train')
    df = ds.to_pandas()
    text_col = next((c for c in ['text','title','content','statement'] if c in df.columns), df.columns[0])
    label_col = next((c for c in ['label','Label','fake','class'] if c in df.columns), df.columns[-1])
    df['text'] = df[text_col].fillna('').str.strip()
    df['label'] = pd.to_numeric(df[label_col], errors='coerce')
    df = df.dropna(subset=['label'])
    df['label'] = df['label'].astype(int)
    df = df[df['label'].isin([0,1])]
    frames.append(df[['text','label']])
    print(f'  OK: {len(df)} rows, labels: {df["label"].value_counts().to_dict()}')
except Exception as e:
    print(f'  Failed: {e}')

# Dataset 3: saurabhp19/fake_news_detection
print('Loading saurabhp19/fake_news_detection...')
try:
    ds = load_dataset('saurabhp19/fake_news_detection', split='train')
    df = ds.to_pandas()
    text_col = next((c for c in ['text','title','statement','content'] if c in df.columns), df.columns[0])
    label_col = next((c for c in ['label','Label','fake'] if c in df.columns), df.columns[-1])
    df['text'] = df[text_col].fillna('').str.strip()
    df['label'] = pd.to_numeric(df[label_col], errors='coerce')
    df = df.dropna(subset=['label'])
    df['label'] = df['label'].astype(int)
    df = df[df['label'].isin([0,1])]
    frames.append(df[['text','label']])
    print(f'  OK: {len(df)} rows, labels: {df["label"].value_counts().to_dict()}')
except Exception as e:
    print(f'  Failed: {e}')

# Dataset 4: difraud
print('Loading difraud/difraud...')
try:
    ds = load_dataset('difraud/difraud', split='train')
    df = ds.to_pandas()
    df['text'] = df['text'].fillna('').str.strip()
    df['label'] = df['label'].astype(int)
    df = df.sample(min(15000, len(df)), random_state=42)
    frames.append(df[['text','label']])
    print(f'  OK: {len(df)} rows, labels: {df["label"].value_counts().to_dict()}')
except Exception as e:
    print(f'  Failed: {e}')

print(f'\nFrames loaded: {len(frames)}')
assert len(frames) > 0, 'No datasets loaded!'

# Merge and clean
df_all = pd.concat(frames, ignore_index=True)
df_all = df_all.dropna(subset=['text','label'])
df_all = df_all[df_all['text'].str.len() >= 20]
df_all['text'] = df_all['text'].str[:512]
df_all = df_all.drop_duplicates(subset=['text'])
df_all['label'] = df_all['label'].astype(int)
df_all = df_all[df_all['label'].isin([0,1])]

print(f'Total after cleaning: {len(df_all)}')
print(f'Label distribution: {df_all["label"].value_counts().to_dict()}')

# Balance classes
fake = df_all[df_all['label']==1]
real = df_all[df_all['label']==0]
target = min(20000, min(len(fake), len(real)))
df_bal = pd.concat([
    fake.sample(target, random_state=42),
    real.sample(target, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Balanced: {len(df_bal)} (fake={df_bal["label"].sum()} real={(df_bal["label"]==0).sum()})')

train_df, temp = train_test_split(df_bal, test_size=0.15, random_state=42, stratify=df_bal['label'])
val_df, test_df = train_test_split(temp, test_size=0.5, random_state=42, stratify=temp['label'])
print(f'Train:{len(train_df)} Val:{len(val_df)} Test:{len(test_df)}')

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

BASE_MODEL = 'microsoft/deberta-v3-base'
MAX_LEN = 256

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=MAX_LEN)

def to_ds(df):
    ds = Dataset.from_dict({'text': df['text'].tolist(), 'labels': df['label'].tolist()})
    return ds.map(tokenize, batched=True, batch_size=256, remove_columns=['text'])

print('Tokenizing...')
train_ds = to_ds(train_df)
val_ds   = to_ds(val_df)
test_ds  = to_ds(test_df)
train_ds.set_format('torch')
val_ds.set_format('torch')
test_ds.set_format('torch')
print('Done')

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
import evaluate

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=2,
    id2label={0:'REAL', 1:'FAKE'},
    label2id={'REAL':0, 'FAKE':1},
    ignore_mismatched_sizes=True
).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

acc_m = evaluate.load('accuracy')
f1_m  = evaluate.load('f1')

def compute_metrics(ep):
    preds = ep[0].argmax(axis=-1)
    return {
        'accuracy': acc_m.compute(predictions=preds, references=ep[1])['accuracy'],
        'f1_macro': f1_m.compute(predictions=preds, references=ep[1], average='macro')['f1']
    }

args = TrainingArguments(
    output_dir='./deberta-factchecker',
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=200,
    lr_scheduler_type='linear',
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    fp16=False,
    bf16=False,
    gradient_accumulation_steps=4,
    max_grad_norm=1.0,
    dataloader_num_workers=2,
    logging_steps=50,
    report_to='none',
    save_total_limit=2,
    seed=42,
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)
print('Trainer ready')

In [ ]:
print('Training (~60-80 min on T4, keep tab open)...')
result = trainer.train()
print(f'Done. Steps:{result.global_step} Loss:{result.training_loss:.4f}')

In [ ]:
from sklearn.metrics import classification_report
from sklearn.metrics import brier_score_loss
import torch.nn.functional as F
import torch, json, datetime, numpy as np

res = trainer.evaluate(test_ds)
print(f'Test Accuracy: {res["eval_accuracy"]:.4f}  F1: {res["eval_f1_macro"]:.4f}')

out = trainer.predict(test_ds)
y_pred = out.predictions.argmax(axis=-1)
y_true = out.label_ids
print(classification_report(y_true, y_pred, target_names=['Real','Fake']))

probs = F.softmax(torch.tensor(out.predictions.astype(np.float32)), dim=-1).numpy()
# Replace any NaN/inf before brier score
fake_probs = np.nan_to_num(probs[:,1], nan=0.5, posinf=1.0, neginf=0.0)
brier = brier_score_loss(y_true, fake_probs)
print(f'Brier score: {brier:.4f}')

import os
os.makedirs('./deberta-factchecker', exist_ok=True)
version = {
    'model': MODEL_NAME, 'base': BASE_MODEL,
    'version': datetime.datetime.utcnow().strftime('%Y%m%d_%H%M'),
    'accuracy': round(float(res['eval_accuracy']),4),
    'f1_macro': round(float(res['eval_f1_macro']),4),
    'brier_score': round(float(brier),4),
    'train_samples': len(train_df), 'max_length': MAX_LEN
}
with open('./deberta-factchecker/model_version.json','w') as f:
    json.dump(version, f, indent=2)
print(json.dumps(version, indent=2))

In [ ]:
if HF_TOKEN:
    print(f'Pushing to {MODEL_NAME}...')
    trainer.push_to_hub(MODEL_NAME)
    tokenizer.push_to_hub(MODEL_NAME)
    print(f'Live at: https://huggingface.co/{MODEL_NAME}')
else:
    trainer.save_model('./deberta-factchecker')
    tokenizer.save_pretrained('./deberta-factchecker')
    print('Saved locally')

from google.colab import files
files.download('./deberta-factchecker/model_version.json')
print('Downloaded model_version.json')
print(f'Set Render env var: DEBERTA_MODEL={MODEL_NAME}')